In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
import os

sys.path.append("benchmarks")

In [3]:
os.getcwd()

'/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB'

In [5]:
from utils.datasets.utils import parse_nt_to_generator, save_from_generator
from utils.datasets.bsbm import BerlinSparqlBenchmark
from pathlib import Path

In [6]:
base_dir = Path("./benchmarks") / "data"

In [7]:
pow = 3
base_dir_bsbm = base_dir / f"bsbm_{pow}"
n = 10**pow
dataset = BerlinSparqlBenchmark(base_dir=base_dir_bsbm, n=n)

base_db_dir = Path("./benchmarks") / "scratch" / "bsbm" / dataset.base_dir.name

In [8]:
from utils.dbs.base_db import BaseDB
from utils.dbs.qlever_native import QleverDBNative
from utils.dbs.qlever import QleverDB


db = QleverDBNative(
    id="timing-qlever",
    base_dir=base_db_dir,
    dataset=dataset,
    use_encoded_ttl=True,
    enable_tensor_index=True,
    name="QLever",
    port_offset=0,
)

2026-11-22 10:43:31,829 - INFO - Loading faiss with AVX512 support.
2026-11-22 10:43:31,830 - INFO - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-11-22 10:43:31,830 - INFO - Loading faiss with AVX2 support.
2026-11-22 10:43:31,830 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-11-22 10:43:31,830 - INFO - Loading faiss.
2026-11-22 10:43:31,858 - INFO - Successfully loaded faiss.
2026-11-22 10:43:31,866 - WARNING - Killing any existing process using port 26043 before starting the server
2026-11-22 10:43:31,876 - ERROR - Command failed with return code 1
2026-11-22 10:43:31,876 - INFO - Initialized QLeverDBNative with id=timing-qlever, port_id=26043, dataset=bsbm, name=QLever, use_encoded_ttl=True, endpoint=http://localhost:26043/timing-qlever-with-tidx/sparql


In [9]:
db.db_dir, db.base_dir

(PosixPath('benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx'),
 PosixPath('benchmarks/scratch/bsbm/bsbm_3'))

In [10]:
db.log_file

PosixPath('logs/QleverDBNative.log')

In [ ]:
with db:
    res = db.query("""SELECT (COUNT(?s) as ?count)
    WHERE {
        ?s rdfs:comment_embedding|rdfs:label_embedding ?o .
    }""")
res

2026-11-22 10:43:31,931 - INFO - Setting up QLeverDBNative db_dir=benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx, base_dir=benchmarks/scratch/bsbm/bsbm_3
2026-11-22 10:43:31,931 - INFO - Logging QLever setup to benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-11-22 10:43:31,931 - INFO - Loading dataset into QLever server from benchmarks/data/bsbm_3/dataset.nt
2026-11-22 10:43:31,932 - WARNING - DB directory benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx already exists!
2026-11-22 10:43:31,932 - INFO - Stopping server!
2026-11-22 10:43:31,936 - ERROR - Command failed with return code 1
2026-11-22 10:43:31,937 - INFO - Starting QLever server on port 26043
2026-11-22 10:43:31,937 - INFO - Running command: qlever-server -i timing-qlever-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-22 10:43:31,941 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refuse

 1537892

26043/tcp:          


,count
0,11840


In [ ]:
with db:
    res = db.query("""
            SELECT DISTINCT ?feature (dta:avg(?vector_comment) AS ?vector_comment_mean) WHERE {
                    ?prod bsbmv:productFeature ?feature ;
                    rdfs:comment_embedding ?vector_comment .
            } GROUP BY ?feature
            LIMIT 10""")
res

2026-11-22 10:43:33,000 - INFO - Setting up QLeverDBNative db_dir=benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx, base_dir=benchmarks/scratch/bsbm/bsbm_3
2026-11-22 10:43:33,000 - INFO - Logging QLever setup to benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-11-22 10:43:33,000 - INFO - Loading dataset into QLever server from benchmarks/data/bsbm_3/dataset.nt
2026-11-22 10:43:33,000 - WARNING - DB directory benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx already exists!
2026-11-22 10:43:33,001 - INFO - Stopping server!
2026-11-22 10:43:33,026 - ERROR - Command failed with return code 1
2026-11-22 10:43:33,027 - INFO - Starting QLever server on port 26043
2026-11-22 10:43:33,027 - INFO - Running command: qlever-server -i timing-qlever-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-22 10:43:33,028 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refuse

2026-11-22 10:43:34,034 - INFO - Server is up and responding to queries
2026-11-22 10:43:34,098 - INFO - Stopping server!


 1537973

26043/tcp:          


,feature,vector_comment_mean
0,bsbmi:ProductFeature1,"{""data"":[0.025642749,-0.058173742,0.031629786,..."
1,bsbmi:ProductFeature10,"{""data"":[0.028887475,-0.057214066,0.027795488,..."
2,bsbmi:ProductFeature100,"{""data"":[0.04539343,-0.055487357,0.035308737,0..."
3,bsbmi:ProductFeature1000,"{""data"":[0.051492184,-0.06205226,0.07084939,-0..."
4,bsbmi:ProductFeature1001,"{""data"":[-0.013997901,-0.045625348,0.045191105..."
5,bsbmi:ProductFeature1002,"{""data"":[0.09144091,-0.062301844,0.03827965,0...."
6,bsbmi:ProductFeature1003,"{""data"":[0.03159111,-0.047059536,0.043941997,0..."
7,bsbmi:ProductFeature1004,"{""data"":[0.06497809,-0.087969266,0.043051764,0..."
8,bsbmi:ProductFeature1005,"{""data"":[0.09144091,-0.062301844,0.03827965,0...."
9,bsbmi:ProductFeature1006,"{""data"":[0.072290026,-0.06485529,0.04107151,0...."


In [ ]:
with db:
    res = db.query("""
            SELECT ?prodA ?featB ?vectorA ?vectorBmean ?dist ?countB
            WHERE {
                ?prodA bsbmv:productFeature ?featB ;
                rdfs:comment_embedding ?vectorA .
                {
                    SELECT ?featB (dta:avg(?vectorB) AS ?vectorBmean) (COUNT(?prodB) AS ?countB) WHERE {
                         ?prodB bsbmv:productFeature ?featB ;
                         rdfs:comment_embedding ?vectorB .
                         FILTER(?featB = bsbmi:ProductFeature1)
                    } GROUP BY ?featB
                }
                FILTER(?countB > 1)
                BIND(dtf:cosineSimilarity(?vectorA, ?vectorBmean) AS ?dist)
            }
            ORDER BY DESC(?dist)
            LIMIT 1""")
res

2026-11-22 10:43:34,142 - INFO - Setting up QLeverDBNative db_dir=benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx, base_dir=benchmarks/scratch/bsbm/bsbm_3
2026-11-22 10:43:34,143 - INFO - Logging QLever setup to benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-11-22 10:43:34,143 - INFO - Loading dataset into QLever server from benchmarks/data/bsbm_3/dataset.nt
2026-11-22 10:43:34,143 - WARNING - DB directory benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx already exists!
2026-11-22 10:43:34,144 - INFO - Stopping server!
2026-11-22 10:43:34,170 - ERROR - Command failed with return code 1
2026-11-22 10:43:34,170 - INFO - Starting QLever server on port 26043
2026-11-22 10:43:34,171 - INFO - Running command: qlever-server -i timing-qlever-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-22 10:43:34,171 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refuse

2026-11-22 10:43:35,178 - INFO - Server is up and responding to queries
2026-11-22 10:43:35,196 - INFO - Stopping server!


 1538036

26043/tcp:          


,prodA,featB,vectorA,vectorBmean,dist,countB
0,bsbmi:dataFromProducer9/Product449,bsbmi:ProductFeature1,"{""data"": [0.03446599096059799, -0.077982082962...","{""data"":[0.025642749,-0.058173742,0.031629786,...",0.9125620722771,63


In [ ]:
with db:
    res = db.query("""
            PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
            SELECT ?prodA ?featB ?vectorA ?vectorBmean ?dist  ?countB
            WHERE {                
                {
                    SELECT ?featB (dta:avg(?vectorB) AS ?vectorBmean) (COUNT(?prodB) AS ?countB) WHERE {
                         ?prodB bsbmv:productFeature ?featB ;
                         rdfs:comment_embedding ?vectorB .

                         FILTER(?featB = bsbmi:ProductFeature1)
                    } GROUP BY ?featB
                }
                SERVICE tensorIndex: {
                    _:config tensorIndex:numNN 1 ;
                    tensorIndex:left ?vectorBmean ;
                    tensorIndex:bindDistance ?dist ;
                    tensorIndex:payload ?prodA, ?featB, ?vectorBmean ;
                    tensorIndex:searchK 16 ;
                    tensorIndex:kIVF 128 ;
                    tensorIndex:experimentalRightCacheName "easy_group_index_dbpedia" ;
                    tensorIndex:right ?vectorA ;
                    tensorIndex:algorithm tensorIndex:ivf ;
                    tensorIndex:distance tensorIndex:dot .
                    {
                        ?prodA bsbmv:productFeature ?featB .       
                        ?prodA rdfs:comment_embedding ?vectorA .   
                         FILTER(?featB = bsbmi:ProductFeature1)
                    }
                }  
                FILTER(?countB > 1)
            }
            LIMIT 10""")
res

2026-11-22 10:43:35,241 - INFO - Setting up QLeverDBNative db_dir=benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx, base_dir=benchmarks/scratch/bsbm/bsbm_3
2026-11-22 10:43:35,241 - INFO - Logging QLever setup to benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-11-22 10:43:35,241 - INFO - Loading dataset into QLever server from benchmarks/data/bsbm_3/dataset.nt
2026-11-22 10:43:35,241 - WARNING - DB directory benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx already exists!
2026-11-22 10:43:35,242 - INFO - Stopping server!


2026-11-22 10:43:35,267 - ERROR - Command failed with return code 1
2026-11-22 10:43:35,268 - INFO - Starting QLever server on port 26043
2026-11-22 10:43:35,268 - INFO - Running command: qlever-server -i timing-qlever-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-22 10:43:35,269 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/timing-qlever-with-tidx/sparql)
2026-11-22 10:43:36,275 - INFO - Server is up and responding to queries
2026-11-22 10:43:36,318 - INFO - Stopping server!


 1538125

26043/tcp:          


,prodA,featB,vectorA,vectorBmean,dist,countB
0,bsbmi:dataFromProducer9/Product449,bsbmi:ProductFeature1,"{""data"": [0.03446599096059799, -0.077982082962...","{""data"":[0.025642749,-0.058173742,0.031629786,...",0.7655559778214,63


In [39]:
with db:
    res = db.query("""
            SELECT ?prodA ?featB ?vectorA ?vectorBmean ?dist ?countB
            WHERE {
                ?prodA bsbmv:productFeature ?featB ;
                rdfs:comment_embedding ?vectorA .
                {
                    SELECT ?featB (dta:avg(?vectorB) AS ?vectorBmean) (COUNT(?prodB) AS ?countB) WHERE {
                         ?prodB bsbmv:productFeature ?featB ;
                         rdfs:comment_embedding ?vectorB .
                    } GROUP BY ?featB
                }
                FILTER(?countB > 1)
                BIND(dtf:cosineSimilarity(?vectorA, ?vectorBmean) AS ?dist)
            }
            ORDER BY DESC(?dist)
            LIMIT 10""")
res

2026-11-22 10:52:48,744 - INFO - Setting up QLeverDBNative db_dir=benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx, base_dir=benchmarks/scratch/bsbm/bsbm_3
2026-11-22 10:52:48,745 - INFO - Logging QLever setup to benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-11-22 10:52:48,745 - INFO - Loading dataset into QLever server from benchmarks/data/bsbm_3/dataset.nt
2026-11-22 10:52:48,745 - WARNING - DB directory benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx already exists!
2026-11-22 10:52:48,745 - INFO - Stopping server!


2026-11-22 10:52:48,771 - ERROR - Command failed with return code 1
2026-11-22 10:52:48,772 - INFO - Starting QLever server on port 26043
2026-11-22 10:52:48,772 - INFO - Running command: qlever-server -i timing-qlever-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-22 10:52:48,773 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/timing-qlever-with-tidx/sparql)
2026-11-22 10:52:49,780 - INFO - Server is up and responding to queries
2026-11-22 10:52:50,006 - INFO - Stopping server!


 1555142

26043/tcp:          


,prodA,featB,vectorA,vectorBmean,dist,countB
0,bsbmi:dataFromProducer6/Product216,bsbmi:ProductFeature2910,"{""data"": [0.028654323890805244, -0.11984816938...","{""data"":[0.028966075,-0.102710284,0.004743871,...",0.9651347994804,2
1,bsbmi:dataFromProducer19/Product877,bsbmi:ProductFeature2910,"{""data"": [0.029277825728058815, -0.08557239919...","{""data"":[0.028966075,-0.102710284,0.004743871,...",0.9651347994804,2
2,bsbmi:dataFromProducer4/Product174,bsbmi:ProductFeature1879,"{""data"": [0.06393353641033173, -0.065314851701...","{""data"":[0.046524942,-0.07222472,0.028981127,0...",0.9619890451431,2
3,bsbmi:dataFromProducer10/Product470,bsbmi:ProductFeature1879,"{""data"": [0.029116343706846237, -0.07913459092...","{""data"":[0.046524942,-0.07222472,0.028981127,0...",0.9619890451431,2
4,bsbmi:dataFromProducer8/Product368,bsbmi:ProductFeature2370,"{""data"": [0.03308946266770363, -0.093641318380...","{""data"":[0.0321147,-0.08266966,0.004120517,0.0...",0.9599935412407,2
5,bsbmi:dataFromProducer11/Product534,bsbmi:ProductFeature2370,"{""data"": [0.03113994002342224, -0.071698009967...","{""data"":[0.0321147,-0.08266966,0.004120517,0.0...",0.9599935412407,2
6,bsbmi:dataFromProducer11/Product511,bsbmi:ProductFeature2536,"{""data"": [-0.023479344323277473, -0.0919813066...","{""data"":[-0.016159674,-0.056431975,0.040700413...",0.9595721364021,2
7,bsbmi:dataFromProducer1/Product38,bsbmi:ProductFeature2536,"{""data"": [-0.008840004913508892, -0.0208826418...","{""data"":[-0.016159674,-0.056431975,0.040700413...",0.9595721364021,2
8,bsbmi:dataFromProducer9/Product442,bsbmi:ProductFeature1885,"{""data"": [0.03240848705172539, -0.105623461306...","{""data"":[0.048171014,-0.08546916,0.025424276,0...",0.9576013684273,2
9,bsbmi:dataFromProducer4/Product174,bsbmi:ProductFeature1885,"{""data"": [0.06393353641033173, -0.065314851701...","{""data"":[0.048171014,-0.08546916,0.025424276,0...",0.9576013684273,2


In [ ]:
with db:
    res = db.query("""
            PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
            SELECT ?prodA ?featB ?vectorA ?vectorBmean ?dist  ?countB
            WHERE {                
                {
                    SELECT ?featB (dta:avg(?vectorB) AS ?vectorBmean) (COUNT(?prodB) AS ?countB) WHERE {
                         ?prodB bsbmv:productFeature ?featB ;
                         rdfs:comment_embedding ?vectorB .
                    } GROUP BY ?featB
                }
                SERVICE tensorIndex: {
                    _:config tensorIndex:numNN 1 ;
                    tensorIndex:left ?vectorBmean ;
                    tensorIndex:bindDistance ?dist ;
                    tensorIndex:payload ?prodA, ?featB, ?vectorBmean ;
                    tensorIndex:searchK 16 ;
                    tensorIndex:kIVF 128 ;
                    tensorIndex:experimentalRightCacheName "easy_group_index_dbpedia" ;
                    tensorIndex:right ?vectorA ;
                    tensorIndex:algorithm tensorIndex:ivf ;
                    tensorIndex:distance tensorIndex:dot .
                    {
                        ?prodA bsbmv:productFeature ?featB .       
                        ?prodA rdfs:comment_embedding ?vectorA .   
                    }
                }  
                FILTER(?countB > 1)
            }
            ORDER BY 
            DESC(?dist)
            LIMIT 10
            """)
res

2026-11-22 10:50:21,946 - INFO - Setting up QLeverDBNative db_dir=benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx, base_dir=benchmarks/scratch/bsbm/bsbm_3
2026-11-22 10:50:21,946 - INFO - Logging QLever setup to benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-11-22 10:50:21,946 - INFO - Loading dataset into QLever server from benchmarks/data/bsbm_3/dataset.nt
2026-11-22 10:50:21,947 - WARNING - DB directory benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx already exists!
2026-11-22 10:50:21,947 - INFO - Stopping server!


2026-11-22 10:50:21,973 - ERROR - Command failed with return code 1
2026-11-22 10:50:21,973 - INFO - Starting QLever server on port 26043
2026-11-22 10:50:21,974 - INFO - Running command: qlever-server -i timing-qlever-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-22 10:50:21,975 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/timing-qlever-with-tidx/sparql)
2026-11-22 10:50:22,981 - INFO - Server is up and responding to queries
2026-11-22 10:50:23,973 - INFO - Stopping server!


 1550623

26043/tcp:          


,prodA,featB,vectorA,vectorBmean,dist,countB
0,bsbmi:dataFromProducer19/Product877,bsbmi:ProductFeature2895,"{""data"": [0.029277825728058815, -0.08557239919...","{""data"":[0.028966075,-0.102710284,0.004743871,...",0.9314850568771,2
1,bsbmi:dataFromProducer4/Product174,bsbmi:ProductFeature13,"{""data"": [0.06393353641033173, -0.065314851701...","{""data"":[0.046524942,-0.07222472,0.028981127,0...",0.925422847271,2
2,bsbmi:dataFromProducer11/Product534,bsbmi:ProductFeature22,"{""data"": [0.03113994002342224, -0.071698009967...","{""data"":[0.0321147,-0.08266966,0.004120517,0.0...",0.9215877056122,2
3,bsbmi:dataFromProducer1/Product38,bsbmi:ProductFeature22,"{""data"": [-0.008840004913508892, -0.0208826418...","{""data"":[-0.016159674,-0.056431975,0.040700413...",0.920778632164,2
4,bsbmi:dataFromProducer9/Product442,bsbmi:ProductFeature10,"{""data"": [0.03240848705172539, -0.105623461306...","{""data"":[0.048171014,-0.08546916,0.025424276,0...",0.9170003533363,2
5,bsbmi:dataFromProducer20/Product923,bsbmi:ProductFeature1258,"{""data"": [0.07749166339635849, -0.050406973809...","{""data"":[0.05974122,-0.06466518,0.023059992,0....",0.9162691831589,2
6,bsbmi:dataFromProducer11/Product537,bsbmi:ProductFeature3337,"{""data"": [0.017261745408177376, -0.05820934846...","{""data"":[0.01796471,-0.05063454,0.027059138,0....",0.9152151346207,2
7,bsbmi:dataFromProducer3/Product120,bsbmi:ProductFeature1258,"{""data"": [0.027751293033361435, -0.04043917357...","{""data"":[0.034871034,-0.059681285,0.027243478,...",0.9142463207245,2
8,bsbmi:dataFromProducer16/Product761,bsbmi:ProductFeature1335,"{""data"": [0.05232652276754379, -0.051554702222...","{""data"":[0.030430451,-0.03965919,0.053464845,0...",0.9137649536133,2
9,bsbmi:dataFromProducer13/Product586,bsbmi:ProductFeature1144,"{""data"": [0.05186239257454872, -0.056225985288...","{""data"":[0.05149892,-0.05489514,0.038708948,0....",0.9123232960701,2


In [55]:
from utils.datasets.base_dataset import QUERY_DIFFICULTY, QUERY_TYPE
from utils.datasets.data_tensor import DataTensor
import numpy as np

with db:
    for type in [QUERY_TYPE.EMBEDDED, QUERY_TYPE.INDEX]:
        for diff in [QUERY_DIFFICULTY.GROUP_EASY, QUERY_DIFFICULTY.GROUP_HARD]:
            res = db.query_auto(
                DataTensor.from_numpy(np.random.rand(128).astype(np.float32)),
                query_difficulty=diff,
                query_type=type,
            )
            print(f"Result for {diff} and {type}: {res.shape} / top-1 {res.iloc[0,0]}")

2026-11-22 11:09:56,025 - INFO - Setting up QLeverDBNative db_dir=benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx, base_dir=benchmarks/scratch/bsbm/bsbm_3
2026-11-22 11:09:56,025 - INFO - Logging QLever setup to benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-11-22 11:09:56,026 - INFO - Loading dataset into QLever server from benchmarks/data/bsbm_3/dataset.nt
2026-11-22 11:09:56,026 - WARNING - DB directory benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx already exists!
2026-11-22 11:09:56,026 - INFO - Stopping server!


2026-11-22 11:09:56,053 - ERROR - Command failed with return code 1
2026-11-22 11:09:56,053 - INFO - Starting QLever server on port 26043
2026-11-22 11:09:56,053 - INFO - Running command: qlever-server -i timing-qlever-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-22 11:09:56,054 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/timing-qlever-with-tidx/sparql)
2026-11-22 11:09:57,060 - INFO - Server is up and responding to queries


Result for QUERY_DIFFICULTY.GROUP_EASY and QUERY_TYPE.EMBEDDED: (10, 6) / top-1 bsbmi:dataFromProducer9/Product449
Result for QUERY_DIFFICULTY.GROUP_HARD and QUERY_TYPE.EMBEDDED: (10, 6) / top-1 bsbmi:dataFromProducer6/Product216
Result for QUERY_DIFFICULTY.GROUP_EASY and QUERY_TYPE.INDEX: (10, 6) / top-1 bsbmi:dataFromProducer9/Product449


2026-11-22 11:09:58,050 - INFO - Stopping server!


Result for QUERY_DIFFICULTY.GROUP_HARD and QUERY_TYPE.INDEX: (10, 6) / top-1 bsbmi:dataFromProducer3/Product115
 1585205

26043/tcp:          


In [56]:
with db:
    qs = db.get_queries(DataTensor.from_numpy(np.random.rand(128).astype(np.float32)))
    q = qs[QUERY_DIFFICULTY.GROUP_EASY][QUERY_TYPE.EMBEDDED]
    res = db.query(q)
res

2026-11-22 11:10:17,321 - INFO - Setting up QLeverDBNative db_dir=benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx, base_dir=benchmarks/scratch/bsbm/bsbm_3
2026-11-22 11:10:17,321 - INFO - Logging QLever setup to benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-11-22 11:10:17,322 - INFO - Loading dataset into QLever server from benchmarks/data/bsbm_3/dataset.nt
2026-11-22 11:10:17,322 - WARNING - DB directory benchmarks/scratch/bsbm/bsbm_3/db/timing-qlever-with-tidx already exists!
2026-11-22 11:10:17,322 - INFO - Stopping server!


2026-11-22 11:10:17,349 - ERROR - Command failed with return code 1
2026-11-22 11:10:17,349 - INFO - Starting QLever server on port 26043
2026-11-22 11:10:17,349 - INFO - Running command: qlever-server -i timing-qlever-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-22 11:10:17,350 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/timing-qlever-with-tidx/sparql)
2026-11-22 11:10:18,357 - INFO - Server is up and responding to queries
2026-11-22 11:10:18,377 - INFO - Stopping server!


 1585979

26043/tcp:          


,productA,featureA,featureB,dist,vectorA,vectorBmean
0,bsbmi:dataFromProducer9/Product449,None,bsbmi:ProductFeature1,0.9125620722771,"{""data"": [0.03446599096059799, -0.077982082962...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
1,bsbmi:dataFromProducer10/Product470,None,bsbmi:ProductFeature1,0.9120101332664,"{""data"": [0.029116343706846237, -0.07913459092...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
2,bsbmi:dataFromProducer16/Product769,None,bsbmi:ProductFeature1,0.9003760814667,"{""data"": [0.050991952419281006, -0.09477660059...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
3,bsbmi:dataFromProducer5/Product191,None,bsbmi:ProductFeature1,0.8994067311287,"{""data"": [0.035210710018873215, -0.06304515153...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
4,bsbmi:dataFromProducer7/Product312,None,bsbmi:ProductFeature1,0.8965536355972,"{""data"": [0.04019315168261528, -0.108818478882...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
5,bsbmi:dataFromProducer12/Product549,None,bsbmi:ProductFeature1,0.8945714831352,"{""data"": [0.0459655299782753, -0.0907632336020...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
6,bsbmi:dataFromProducer16/Product751,None,bsbmi:ProductFeature1,0.8903938531876,"{""data"": [0.04121268913149834, -0.050871748477...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
7,bsbmi:dataFromProducer3/Product106,None,bsbmi:ProductFeature1,0.8868094086647,"{""data"": [0.02746967226266861, -0.103566497564...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
8,bsbmi:dataFromProducer2/Product80,None,bsbmi:ProductFeature1,0.8866115808487,"{""data"": [0.08607114106416702, -0.077222004532...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
9,bsbmi:dataFromProducer17/Product799,None,bsbmi:ProductFeature1,0.8857095837593,"{""data"": [0.011794944293797016, -0.03868792206...","{""data"":[0.025642749,-0.058173742,0.031629786,..."
